# 01 – Exploratory Data Analysis (EDA)

This notebook downloads NASDAQ-100 data, inspects its structure, and creates
exploratory visualisations to understand the data before modelling.

In [33]:
import sys
sys.path.insert(0, '..')

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

from src.config import TICKER, START_DATE, END_DATE
from src.data.collector import DataCollector
from src.visualization.plotter import Plotter

print(f'Ticker : {TICKER}')
print(f'Period : {START_DATE} → {END_DATE}')

Ticker : ^NDX
Period : 2020-02-01 → 2025-12-31


In [23]:
# ── 1. Download stock data ──────────────────────────────────────────────────
collector = DataCollector()
df = collector.download_stock_data(save=True)
print(f'Shape : {df.shape}')
df.head()

Shape : (1486, 6)


Price,Adj Close,Close,High,Low,Open,Volume
Date,,,,,,
2020-02-03,9126.230469,9126.230469,9148.500000,9031.070312,9033.519531,2427320000
2020-02-04,9334.059570,9334.059570,9352.959961,9224.860352,9256.940430,2447340000
2020-02-05,9367.480469,9367.480469,9442.750000,9311.879883,9441.339844,2470240000
2020-02-06,9445.919922,9445.919922,9448.719727,9357.860352,9395.690430,2313860000
2020-02-07,9401.099609,9401.099609,9453.240234,9376.910156,9397.769531,2243720000


In [24]:
# ── 2. Basic statistics ─────────────────────────────────────────────────────
df.describe()

Price,Adj Close,Close,High,Low,Open,Volume
count,1486.000000,1486.000000,1486.000000,1486.000000,1486.000000,1.486000e+03
mean,15637.653004,15637.653004,15751.246618,15507.784891,15634.115863,5.666445e+09
std,4324.188704,4324.188704,4330.512912,4314.310126,4327.734741,1.921232e+09
min,6994.290039,6994.290039,7145.290039,6771.910156,6952.709961,2.184080e+09
25%,12343.490234,12343.490234,12467.882324,12217.370117,12353.814941,4.417512e+09
50%,14844.544922,14844.544922,14965.589844,14737.560059,14866.064941,5.055150e+09
75%,18829.958008,18829.958008,19030.154297,18666.612793,18832.475586,6.337338e+09
max,26119.849609,26119.849609,26182.099609,25907.449219,26147.720703,1.630873e+10


In [25]:
# ── 3. Missing values ───────────────────────────────────────────────────────
print('Missing values:')
print(df.isnull().sum())

Missing values:
Price
Adj Close    0
Close        0
High         0
Low          0
Open         0
Volume       0
dtype: int64


In [29]:
# ── 4. Price history plot ───────────────────────────────────────────────────
print("START_DATE =", START_DATE)
print("END_DATE   =", END_DATE)
print("df.index min/max:", df.index.min(), df.index.max())
df_plot = df.loc[START_DATE:END_DATE].copy()
plotter = Plotter()
plotter.plot_price_history(df_plot, title=f"{TICKER} Closing Price", filename="price_history.png")

START_DATE = 2020-02-01
END_DATE   = 2025-12-31
df.index min/max: 2020-02-03 00:00:00 2025-12-30 00:00:00


In [6]:
# ── 5. Returns distribution ─────────────────────────────────────────────────
returns = df['Close'].pct_change().dropna()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(returns.index, returns, linewidth=0.5, color='steelblue')
axes[0].set_title('Daily Returns')
axes[0].set_ylabel('Return')
axes[0].grid(alpha=0.3)

axes[1].hist(returns, bins=80, color='steelblue', edgecolor='white')
axes[1].set_title('Return Distribution')
axes[1].set_xlabel('Daily Return')
axes[1].grid(alpha=0.3)

plt.tight_layout()
fig.savefig('../reports/figures/returns_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'Skewness : {returns.skew():.4f}')
print(f'Kurtosis : {returns.kurtosis():.4f}')

Skewness : -0.1149
Kurtosis : 7.1135


In [7]:
# ── 6. Rolling volatility (30-day) ──────────────────────────────────────────
vol = returns.rolling(30).std() * (252 ** 0.5)  # annualised

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(vol.index, vol, color='tomato', linewidth=0.8)
ax.set_title('30-Day Rolling Annualised Volatility')
ax.set_ylabel('Volatility')
ax.grid(alpha=0.3)
fig.autofmt_xdate()
fig.savefig('../reports/figures/rolling_volatility.png', dpi=150, bbox_inches='tight')
plt.show()

In [12]:
# ── 7. Volume analysis ──────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(12, 3))
ax.bar(df.index, df['Volume'], width=1, color='grey', alpha=0.6)
ax.set_title('Daily Trading Volume')
ax.set_ylabel('Volume')
ax.grid(alpha=0.3)
fig.autofmt_xdate()
plt.tight_layout()
fig.savefig('../reports/figures/trading_volume.png', dpi=150, bbox_inches='tight')
plt.show()